In [0]:
# Cell 1 — Setup: base path + all table names
# Run this first every session — establishes all variables

base_path = "abfss://adventureworks@adwsadatabricksdev.dfs.core.windows.net"

# Dictionary of all our source tables
# Key = friendly table name, Value = CSV filename in ADLS
tables = {
    "salesterritory"      : "salesterritory.csv",
    "salesorderheader"    : "salesorderheader.csv",
    "salesorderdetail"    : "salesorderdetail.csv",
    "customer"            : "customer.csv",
    "person"              : "person.csv",
    "product"             : "product.csv",
    "productcategory"     : "productcategory.csv",
    "productsubcategory"  : "productsubcategory.csv",
    "productinventory"    : "productinventory.csv"
}

print("✅ Setup complete!")
print(f"   Base path : {base_path}")
print(f"   Tables    : {len(tables)} source tables registered")

In [0]:
# Cell 2 — Load all 9 tables into DataFrames in one loop
# This is a production pattern — never repeat yourself when you can loop

dataframes = {}  # Empty dictionary to store our DataFrames

for table_name, csv_file in tables.items():
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{base_path}/{csv_file}")
    
    dataframes[table_name] = df  # Store each DataFrame by table name
    print(f"✅ Loaded {table_name:<25} | Rows: {df.count():>7,} | Columns: {len(df.columns)}")

print(f"\n🎉 All {len(dataframes)} tables loaded successfully!")

In [0]:
# Cell 3 — Print schema for every table
# This tells us column names and data types for all 9 tables
# SQL equivalent: sp_help 'TableName' in SSMS

for table_name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"  📋 TABLE: {table_name.upper()}")
    print(f"{'='*60}")
    print(f"  Rows: {df.count():,} | Columns: {len(df.columns)}")
    print(f"  Columns:")
    for col_name, col_type in df.dtypes:
        print(f"    |-- {col_name:<35} {col_type}")

In [0]:
# Cell 4 — Null analysis for every table
# SQL equivalent: SELECT COUNT(*) - COUNT(column_name) FROM table
# This tells us which columns have missing data

from pyspark.sql import functions as F

for table_name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"  🔍 NULL ANALYSIS: {table_name.upper()}")
    print(f"{'='*60}")
    
    total_rows = df.count()
    
    # Count nulls for every column
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ]).collect()[0]
    
    has_nulls = False
    for col_name in df.columns:
        null_count = null_counts[col_name]
        if null_count > 0:
            pct = round((null_count / total_rows) * 100, 1)
            print(f"  ⚠️  {col_name:<35} {null_count:>6,} nulls ({pct}%)")
            has_nulls = True
    
    if not has_nulls:
        print(f"  ✅ No nulls found in any column!")

In [0]:
# Cell 5 — Duplicate analysis for every table
# SQL equivalent: 
# SELECT COUNT(*) - COUNT(DISTINCT *) FROM table

for table_name, df in dataframes.items():
    total_rows    = df.count()
    distinct_rows = df.distinct().count()
    duplicate_count = total_rows - distinct_rows
    
    status = "✅ No duplicates" if duplicate_count == 0 else f"⚠️  {duplicate_count:,} duplicates found"
    
    print(f"{table_name:<25} | Total: {total_rows:>7,} | Distinct: {distinct_rows:>7,} | {status}")

In [0]:
# Cell 6 — Clean summary dashboard of all tables
# Quick reference for entire source dataset

print(f"\n{'='*65}")
print(f"  📊 ADVENTUREWORKS SOURCE DATA SUMMARY")
print(f"{'='*65}")
print(f"  {'Table':<25} {'Rows':>10} {'Columns':>10} {'Domain':<15}")
print(f"  {'-'*55}")

domain_map = {
    "salesterritory"     : "Sales",
    "salesorderheader"   : "Sales",
    "salesorderdetail"   : "Sales",
    "customer"           : "Person",
    "person"             : "Person",
    "product"            : "Production",
    "productcategory"    : "Production",
    "productsubcategory" : "Production",
    "productinventory"   : "Production"
}

total_rows_all = 0
for table_name, df in dataframes.items():
    row_count = df.count()
    total_rows_all += row_count
    domain = domain_map[table_name]
    print(f"  {table_name:<25} {row_count:>10,} {len(df.columns):>10} {domain:<15}")

print(f"  {'-'*55}")
print(f"  {'TOTAL':<25} {total_rows_all:>10,}")
print(f"{'='*65}")

In [0]:
# Cell 7 — Validate key relationships between tables
# This confirms our joins will work correctly in Silver layer

print("🔗 KEY RELATIONSHIP CHECKS\n")

# Check 1: Do all SalesOrderDetail records have a matching SalesOrderHeader?
df_header = dataframes["salesorderheader"]
df_detail = dataframes["salesorderdetail"]

header_ids = df_header.select("SalesOrderID").distinct()
detail_ids  = df_detail.select("SalesOrderID").distinct()

orphan_details = detail_ids.subtract(header_ids).count()
print(f"✅ SalesOrderDetail → SalesOrderHeader")
print(f"   Orphaned detail rows (no matching header): {orphan_details}")

# Check 2: Do all Products in OrderDetail exist in Product table?
df_product = dataframes["product"]
product_ids         = df_product.select("ProductID").distinct()
detail_product_ids  = df_detail.select("ProductID").distinct()

orphan_products = detail_product_ids.subtract(product_ids).count()
print(f"\n✅ SalesOrderDetail → Product")
print(f"   Products in orders with no product record: {orphan_products}")

# Check 3: SalesTerritory coverage
df_customer = dataframes["customer"]
territory_ids          = dataframes["salesterritory"].select("TerritoryID").distinct()
customer_territory_ids = df_customer.select("TerritoryID").distinct()

unmatched = customer_territory_ids.subtract(territory_ids).count()
print(f"\n✅ Customer → SalesTerritory")
print(f"   Customers with unmatched TerritoryID: {unmatched}")

In [0]:
# Cell 8 — Quick preview of every table
# SQL equivalent: SELECT TOP 3 * FROM each table

for table_name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"  👀 PREVIEW: {table_name.upper()}")
    print(f"{'='*60}")
    df.show(3, truncate=True)

In [0]:
# Cell 9 Fixed — Sales Date Range Analysis
# First let's see the raw date format coming from CSV

df_header = dataframes["salesorderheader"]

# Step 1 - Check raw date values first
print("🔍 RAW DATE FORMAT CHECK:")
df_header.select("OrderDate").show(5, truncate=False)

# Step 2 - Date range using string-safe functions
print("\n📅 SALES DATE RANGE ANALYSIS")
print("="*45)

df_header.select(
    F.min("OrderDate").alias("Earliest_Order"),
    F.max("OrderDate").alias("Latest_Order"),
    F.countDistinct("CustomerID").alias("Unique_Customers"),
    F.countDistinct("TerritoryID").alias("Territories_Covered")
).show(truncate=False)

# Step 3 - Orders by year using substring (works on any date string format)
print("\n📊 ORDERS BY YEAR:")
df_header.withColumn("Year", F.substring("OrderDate", 1, 4)) \
    .groupBy("Year") \
    .agg(
        F.count("SalesOrderID").alias("Total_Orders"),
        F.round(F.sum("TotalDue"), 2).alias("Total_Revenue")
    ) \
    .orderBy("Year") \
    .show()

In [0]:
# Cell 10 — Understand product catalog structure
# This shows how products are organized across categories

df_prod     = dataframes["product"]
df_subcat   = dataframes["productsubcategory"]
df_cat      = dataframes["productcategory"]

print("🏷️  PRODUCT CATALOG STRUCTURE")
print("="*45)

# Join all three product tables together
df_product_full = df_prod \
    .join(df_subcat, "ProductSubcategoryID", "left") \
    .join(df_cat, "ProductCategoryID", "left")

# Count products per category
df_product_full \
    .groupBy(df_cat["Name"].alias("Category")) \
    .agg(F.count("ProductID").alias("Product_Count")) \
    .orderBy("Product_Count", ascending=False) \
    .show()

In [0]:
# Cell 10 — Understand product catalog structure
# This shows how products are organized across categories

df_prod     = dataframes["product"]
df_subcat   = dataframes["productsubcategory"]
df_cat      = dataframes["productcategory"]

print("🏷️  PRODUCT CATALOG STRUCTURE")
print("="*45)

# Clean CSV placeholder values before joining
prod_clean = df_prod.withColumn(
    "ProductSubcategoryID_int",
    F.when(F.col("ProductSubcategoryID") == "NULL", None)
     .otherwise(F.col("ProductSubcategoryID"))
     .cast("int")
)

# Join all three product tables together
df_product_full = prod_clean.alias("p") \
    .join(df_subcat.alias("s"), F.col("p.ProductSubcategoryID_int") == F.col("s.ProductSubcategoryID"), "left") \
    .join(df_cat.alias("c"), F.col("s.ProductCategoryID") == F.col("c.ProductCategoryID"), "left")

# Count products per category
df_product_full \
    .groupBy(F.coalesce(F.col("c.Name"), F.lit("Uncategorized")).alias("Category")) \
    .agg(F.count("p.ProductID").alias("Product_Count")) \
    .orderBy(F.desc("Product_Count")) \
    .show()